[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/en_clase/S11_02_transfer_learning_en_clase.ipynb)

# S11 · Práctica 2 — *Transfer learning* y *fine-tuning* · versión en clase

**Tiempo estimado:** 75 min en clase · **Sprint 11** (UD 4.5)

📖 *¿El código se hace largo? Léelo [explicado por escrito](https://github.com/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/codigo_explicado/S11_02_transfer_learning.md): recorrido, glosario y qué hace cada celda.* <!--codigo-explicado-->

**Contexto TurisData.** Entrenar una CNN desde cero necesita miles de fotos etiquetadas, y el cliente tiene
solo unos cientos. La solución habitual es **reutilizar una red ya entrenada** en otra tarea y adaptarla:
es el *transfer learning*.

**Al terminar sabrás:**
1. Qué se reutiliza (la "base" convolucional) y qué se reentrena (la "cabeza" clasificadora).
2. Congelar capas, entrenar una cabeza nueva y hacer *fine-tuning* de las últimas capas.
3. Medir cuánto ayuda el *transfer learning* cuando hay pocos datos.
4. Cómo se haría con pesos reales (MobileNetV2 con ImageNet) y qué cambia.

**Dos ramas.**
- **Rama offline (ejecutable en el taller):** como no hay internet ni pesos de ImageNet, el propio notebook
  **preentrena una red pequeña** en una tarea auxiliar sintética (formas y texturas) y después la reutiliza para reconocer
  tipos de alojamiento **esquemáticos** con pocas imágenes. Demuestra la **mecánica** completa.
- **Rama Colab (requiere internet, sección 6):** carga de `MobileNetV2` con pesos de ImageNet. **No verificada** en el taller.

> 🏫 **Versión «en clase»** · ⏱ ≈ 75 min <!--reparto-->
>
> - Es **la parte de este cuaderno que se hace en el aula**, con el docente: ejercicios **1**.
> - Lo demás (ejercicios 🏠 y mini-reto) está en el [cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/para_casa/S11_02_transfer_learning.ipynb), para hacer en casa.


In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import functools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)

## 0. Generador de imágenes sintéticas · ▶️ ejecutar y leer
Estas funciones dibujan con PIL (i) **formas y texturas** (para el preentrenamiento) y (ii) **alojamientos
esquemáticos** (la tarea real). No hace falta que las estudies: ejecútala y sigue. Son dibujos, **no fotos reales**.

In [ ]:
from PIL import Image, ImageDraw

CLASES_ALOJ = ["hotel", "apartamentos", "casa_rural", "camping", "piscina"]
CLASES_FORMAS = ["rayas_horizontales", "rayas_verticales", "diagonales", "cuadricula", "puntos", "circulo", "rectangulo", "triangulo"]


def _color(rng, base, var=30):
    """Color aleatorio cercano a `base` (tupla RGB)."""
    return tuple(int(np.clip(c + rng.integers(-var, var + 1), 0, 255)) for c in base)


def _final(img, rng, ruido=8):
    """Convierte a uint8 con un poco de ruido y variación de brillo."""
    a = np.asarray(img, dtype="float32") * rng.uniform(0.85, 1.15)
    a = a + rng.normal(0, ruido, a.shape)
    return np.clip(a, 0, 255).astype("uint8")


def _ventanas(d, x0, y0, x1, y1, filas, cols, color, margen):
    """Cuadrícula de ventanas dentro del rectángulo (x0, y0, x1, y1)."""
    ancho, alto = (x1 - x0) / cols, (y1 - y0) / filas
    for f in range(filas):
        for c in range(cols):
            a, b = x0 + c * ancho + margen * ancho, y0 + f * alto + margen * alto
            d.rectangle([a, b, a + ancho * (1 - 2 * margen), b + alto * (1 - 2 * margen)], fill=color)


def dibujar_alojamiento(clase, rng, tam=48, ruido=8):
    """Dibujo esquemático (tam×tam×3, uint8) de un tipo de alojamiento. Datos SINTÉTICOS, no fotos reales."""
    E = 3                                   # supermuestreo para suavizar bordes
    S = tam * E
    img = Image.new("RGB", (S, S), _color(rng, (170, 205, 235), 25))          # cielo
    d = ImageDraw.Draw(img)
    if rng.random() < 0.5:                                                    # sol opcional
        cx, cy, r = rng.uniform(0.1, 0.9) * S, rng.uniform(0.08, 0.25) * S, rng.uniform(0.05, 0.08) * S
        d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(250, 235, 120))
    hor = rng.uniform(0.62, 0.72) * S                                         # línea del horizonte
    verde = _color(rng, (120, 170, 90), 25)
    if clase == "piscina":
        d.rectangle([0, hor * 0.6, S, S], fill=_color(rng, (225, 210, 170), 15))   # terraza
        x0, x1 = rng.uniform(0.08, 0.2) * S, rng.uniform(0.75, 0.92) * S
        y0, y1 = rng.uniform(0.5, 0.58) * S, rng.uniform(0.82, 0.94) * S
        d.rectangle([x0 - 3, y0 - 3, x1 + 3, y1 + 3], fill=(245, 245, 245))
        d.rectangle([x0, y0, x1, y1], fill=_color(rng, (60, 150, 210), 20))
        for k in range(3):
            yy = y0 + (k + 1) * (y1 - y0) / 4
            d.line([x0 + 6, yy, x1 - 6, yy], fill=(150, 210, 240), width=2)
        for k in range(2):                                                    # tumbonas
            tx = rng.uniform(0.1, 0.8) * S
            d.rectangle([tx, hor * 0.72, tx + 0.1 * S, hor * 0.72 + 0.04 * S], fill=_color(rng, (240, 120, 60), 30))
    else:
        d.rectangle([0, hor, S, S], fill=verde)                                # césped
    if clase == "hotel":
        w, h = rng.uniform(0.30, 0.42) * S, rng.uniform(0.5, 0.66) * S
        x0 = rng.uniform(0.1 * S, 0.9 * S - w); y0 = hor - h + 0.05 * S
        d.rectangle([x0, y0, x0 + w, hor + 0.05 * S], fill=_color(rng, (220, 200, 170)))
        _ventanas(d, x0, y0 + 0.02 * S, x0 + w, hor, int(rng.integers(4, 6)), 3, (60, 90, 140), 0.22)
    elif clase == "apartamentos":
        w, h = rng.uniform(0.7, 0.86) * S, rng.uniform(0.28, 0.38) * S
        x0 = rng.uniform(0.02 * S, 0.98 * S - w); y0 = hor - h + 0.05 * S
        d.rectangle([x0, y0, x0 + w, hor + 0.05 * S], fill=_color(rng, (235, 225, 205)))
        _ventanas(d, x0, y0, x0 + w, hor + 0.05 * S, 2, int(rng.integers(5, 7)), (70, 100, 150), 0.25)
        for k in (1, 2):                                                       # balcones (líneas horizontales)
            yy = y0 + k * (hor + 0.05 * S - y0) / 2
            d.line([x0, yy, x0 + w, yy], fill=(120, 110, 100), width=2)
    elif clase == "casa_rural":
        w, h = rng.uniform(0.32, 0.44) * S, rng.uniform(0.2, 0.28) * S
        x0 = rng.uniform(0.1 * S, 0.9 * S - w); y0 = hor - h + 0.05 * S
        d.rectangle([x0, y0, x0 + w, hor + 0.05 * S], fill=_color(rng, (235, 225, 200)))
        d.polygon([(x0 - 0.04 * S, y0), (x0 + w + 0.04 * S, y0), (x0 + w / 2, y0 - rng.uniform(0.14, 0.2) * S)],
                  fill=_color(rng, (170, 70, 50), 25))
        d.rectangle([x0 + w * 0.42, y0 + h * 0.4, x0 + w * 0.58, hor + 0.05 * S], fill=(110, 70, 40))
        d.rectangle([x0 + w * 0.1, y0 + h * 0.2, x0 + w * 0.3, y0 + h * 0.5], fill=(70, 100, 150))
    elif clase == "camping":
        d.rectangle([0, hor - 0.08 * S, S, S], fill=verde)
        w, h = rng.uniform(0.34, 0.5) * S, rng.uniform(0.28, 0.4) * S
        x0 = rng.uniform(0.05 * S, 0.95 * S - w); base = hor + 0.1 * S
        c = _color(rng, [(240, 140, 40), (60, 120, 200), (230, 200, 60)][int(rng.integers(0, 3))], 20)
        d.polygon([(x0, base), (x0 + w, base), (x0 + w / 2, base - h)], fill=c)
        d.polygon([(x0 + w * 0.4, base), (x0 + w * 0.6, base), (x0 + w / 2, base - h * 0.5)], fill=(50, 40, 40))
    if clase != "piscina" and rng.random() < 0.6:                              # árbol de adorno
        tx = rng.choice([rng.uniform(0.03, 0.12), rng.uniform(0.88, 0.97)]) * S
        d.rectangle([tx - 2, hor - 0.02 * S, tx + 2, hor + 0.12 * S], fill=(100, 70, 40))
        d.ellipse([tx - 0.07 * S, hor - 0.16 * S, tx + 0.07 * S, hor + 0.01 * S], fill=(50, 120, 60))
    return _final(img.resize((tam, tam), Image.LANCZOS), rng, ruido)


def dibujar_forma(clase, rng, tam=48):
    """Formas y texturas abstractas (tarea auxiliar de preentrenamiento). Devuelve tam×tam×3 uint8."""
    fondo, tinta = _color(rng, (rng.integers(20, 235),) * 3, 20), _color(rng, tuple(int(v) for v in rng.integers(0, 256, 3)), 10)
    if np.abs(np.array(fondo, dtype=int) - np.array(tinta, dtype=int)).sum() < 150:      # asegura contraste
        tinta = tuple(255 - c for c in fondo)
    E = 2
    S = tam * E
    img = Image.new("RGB", (S, S), fondo)
    d = ImageDraw.Draw(img)
    paso = int(rng.integers(6, 12)) * E
    off = int(rng.integers(0, paso))
    if clase == "rayas_horizontales":
        for y in range(off, S, paso): d.rectangle([0, y, S, y + paso // 2], fill=tinta)
    elif clase == "rayas_verticales":
        for x in range(off, S, paso): d.rectangle([x, 0, x + paso // 2, S], fill=tinta)
    elif clase == "diagonales":
        for k in range(-S, S, paso): d.line([k + off, 0, k + off + S, S], fill=tinta, width=max(2, paso // 3))
    elif clase == "cuadricula":
        for y in range(off, S, paso): d.line([0, y, S, y], fill=tinta, width=2)
        for x in range(off, S, paso): d.line([x, 0, x, S], fill=tinta, width=2)
    elif clase == "puntos":
        r = max(2, paso // 4)
        for y in range(off, S, paso):
            for x in range(off, S, paso): d.ellipse([x - r, y - r, x + r, y + r], fill=tinta)
    else:
        r = rng.uniform(0.18, 0.38) * S
        cx, cy = rng.uniform(r, S - r), rng.uniform(r, S - r)
        if clase == "circulo": d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=tinta)
        elif clase == "rectangulo": d.rectangle([cx - r, cy - r * rng.uniform(0.5, 1), cx + r, cy + r * rng.uniform(0.5, 1)], fill=tinta)
        else: d.polygon([(cx - r, cy + r), (cx + r, cy + r), (cx, cy - r)], fill=tinta)
    return _final(img.resize((tam, tam), Image.LANCZOS), rng)


def generar_lote(funcion, clases, n_por_clase, semilla=42, tam=48):
    """Devuelve (X uint8 [n, tam, tam, 3], y int [n]) con n_por_clase imágenes por clase, mezcladas."""
    rng = np.random.default_rng(semilla)
    X = np.stack([funcion(c, rng, tam) for c in clases for _ in range(n_por_clase)])
    y = np.repeat(np.arange(len(clases)), n_por_clase)
    orden = rng.permutation(len(y))
    return X[orden], y[orden]


PARECIDAS = {"hotel": "apartamentos", "apartamentos": "hotel", "casa_rural": "camping", "camping": "casa_rural", "piscina": "camping"}


def crear_dataset_carpetas(carpeta, n_por_clase=100, semilla=7, tam=48, ruido=8, prop_ambiguas=0.0):
    """Escribe PNG en carpeta/<clase>/img_000.png (estructura que espera image_dataset_from_directory).

    prop_ambiguas: fracción de imágenes cuyo dibujo corresponde a una clase parecida (PARECIDAS) pero se guarda
    en la carpeta de la otra: simula etiquetas dudosas o erróneas. Se anotan en carpeta/_ambiguas.csv (solo docente).
    """
    import os
    rng = np.random.default_rng(semilla)
    ambiguas = []
    for clase in CLASES_ALOJ:
        os.makedirs(os.path.join(carpeta, clase), exist_ok=True)
        for i in range(n_por_clase):
            dibujo = PARECIDAS[clase] if rng.random() < prop_ambiguas else clase
            if dibujo != clase:
                ambiguas.append(f"{clase}/img_{i:03d}.png,{dibujo}")
            Image.fromarray(dibujar_alojamiento(dibujo, rng, tam, ruido)).save(os.path.join(carpeta, clase, f"img_{i:03d}.png"))
    with open(os.path.join(carpeta, "_ambiguas.csv"), "w") as f:
        f.write("archivo,dibujo_real\n" + "\n".join(ambiguas) + "\n")
    return carpeta

In [ ]:
X_f, y_f = generar_lote(dibujar_forma, CLASES_FORMAS, 6, semilla=1)
X_a, y_a = generar_lote(dibujar_alojamiento, CLASES_ALOJ, 6, semilla=1)
fig, ejes = plt.subplots(2, 10, figsize=(14, 3.2))
for i in range(10):
    ejes[0, i].imshow(X_f[i]); ejes[0, i].set_title(CLASES_FORMAS[y_f[i]], fontsize=7); ejes[0, i].axis("off")
    ejes[1, i].imshow(X_a[i]); ejes[1, i].set_title(CLASES_ALOJ[y_a[i]], fontsize=7); ejes[1, i].axis("off")
plt.suptitle("Arriba: tarea auxiliar (formas y texturas) · Abajo: tarea real (tipos de alojamiento esquemáticos)")
plt.tight_layout(); plt.show()

## 1. La tarea real, con pocos datos

**Idea.** Imagina que TurisData solo tiene **6 imágenes por tipo de alojamiento** (30 en total) etiquetadas y con mucho ruido.
Es muy poco para una CNN desde cero. Reservamos 200 para validación y 300 para prueba.

In [ ]:
N_POR_CLASE = 6
alojamiento = functools.partial(dibujar_alojamiento, ruido=40)     # imágenes "sucias": mucho ruido, como fotos de mala calidad
X_ent, y_ent = generar_lote(alojamiento, CLASES_ALOJ, N_POR_CLASE, semilla=10)
X_val, y_val = generar_lote(alojamiento, CLASES_ALOJ, 40, semilla=11)
X_pru, y_pru = generar_lote(alojamiento, CLASES_ALOJ, 60, semilla=12)
print("entrenamiento", X_ent.shape, "· validación", X_val.shape, "· prueba", X_pru.shape)

## 2. Punto de partida: una CNN desde cero · ▶️ ejecutar y leer
La red se compone de una **base** (bloques `Conv → Pool` que extraen rasgos y terminan en
`Flatten`, que aplana los mapas en una lista de rasgos) y una **cabeza** (capa `Dense` que decide la clase).
Esta división es la clave del *transfer learning*: **la base se reutiliza; la cabeza se cambia**.

In [ ]:
def construir_base(nombre="base"):
    entrada = keras.layers.Input(shape=(48, 48, 3))
    x = keras.layers.Rescaling(1 / 255)(entrada)                     # la propia red normaliza los píxeles
    for i, f in enumerate((16, 32, 64), start=1):
        x = keras.layers.Conv2D(f, 3, padding="same", activation="relu", name=f"bloque{i}_conv")(x)
        x = keras.layers.MaxPooling2D(name=f"bloque{i}_pool")(x)
    x = keras.layers.Flatten(name="aplanar")(x)                       # 6×6×64 = 2304 rasgos
    return keras.Model(entrada, x, name=nombre)

def con_cabeza(base, n_clases, dropout=0.2, tasa=1e-3):
    """Base + cabeza nueva (Dropout y Dense softmax), ya compilado."""
    entrada = keras.layers.Input(shape=(48, 48, 3))
    x = base(entrada)
    x = keras.layers.Dropout(dropout)(x)
    salida = keras.layers.Dense(n_clases, activation="softmax", name="cabeza")(x)
    modelo = keras.Model(entrada, salida)
    modelo.compile(keras.optimizers.Adam(tasa), "sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo

def entrenar(modelo, epocas=30, semilla=SEMILLA, X=None, y=None):
    keras.utils.set_random_seed(semilla)
    X = X_ent if X is None else X
    y = y_ent if y is None else y
    return modelo.fit(X, y, validation_data=(X_val, y_val), epochs=epocas, batch_size=16, verbose=0)

keras.utils.set_random_seed(SEMILLA)
desde_cero = con_cabeza(construir_base("base_desde_cero"), 5)
h_cero = entrenar(desde_cero, epocas=40)
acc_cero = desde_cero.evaluate(X_pru, y_pru, verbose=0)[1]
print(f"CNN desde cero con {len(X_ent)} imágenes → exactitud en prueba: {acc_cero:.3f}")

## 3. Preentrenar la base en una tarea auxiliar

En la vida real se descargaría una base entrenada con **millones de fotos** (ImageNet). Aquí, para poder trabajar sin
internet, la base aprende primero una tarea **distinta pero relacionada**: reconocer 8 clases de formas y
texturas (rayas, cuadrículas, círculos, triángulos…) con 3200 imágenes. Esos rasgos (bordes, esquinas, patrones
repetidos) también sirven para reconocer edificios y tiendas de campaña.

In [ ]:
X_aux, y_aux = generar_lote(dibujar_forma, CLASES_FORMAS, 400, semilla=3)
X_aux_v, y_aux_v = generar_lote(dibujar_forma, CLASES_FORMAS, 50, semilla=4)

keras.utils.set_random_seed(SEMILLA)
base_pre = construir_base("base_preentrenada")
modelo_aux = con_cabeza(base_pre, len(CLASES_FORMAS), dropout=0.0)
modelo_aux.fit(X_aux, y_aux, epochs=12, batch_size=64, verbose=0)
print("Exactitud en la tarea auxiliar (validación):", round(modelo_aux.evaluate(X_aux_v, y_aux_v, verbose=0)[1], 3))
print("La base tiene", base_pre.count_params(), "parámetros y ya sabe extraer rasgos visuales.")

## 4. Congelar la base y entrenar solo una cabeza nueva

**Idea.** "Congelar" una capa (`trainable = False`) significa que su peso **no se actualiza** al entrenar. Así
conservamos lo aprendido y solo se entrenan las pocas neuronas de la cabeza: hacen falta menos datos y menos tiempo.

**Ejercicio 1 · 🏫 en clase.** Escribe `congelar(modelo)` que ponga `trainable = False` en **todas** las capas del modelo dado
(es decir, `modelo.trainable = False`) y lo devuelva. Después crea `copia_base` (una copia independiente de
`base_pre`, ya lo tienes hecho abajo) y congélala.

In [ ]:
# TODO: modelo.trainable = False y return modelo
...

def copiar_base(base, nombre):
    """Copia independiente de la base con los mismos pesos (para no modificar la original)."""
    copia = keras.models.clone_model(base)
    copia.set_weights(base.get_weights())
    copia._name = nombre
    return copia

copia_base = congelar(copiar_base(base_pre, "base_congelada"))
assert len(copia_base.trainable_weights) == 0, "Tras congelar no debe quedar ningún peso entrenable en la base"
assert len(copia_base.non_trainable_weights) == len(base_pre.weights), "Todos los pesos de la base deben pasar a 'no entrenables'"
assert len(base_pre.trainable_weights) > 0, "La base original no debe quedar congelada (usa una copia)"

In [ ]:
transfer = con_cabeza(copia_base, 5)
n_entrenables = sum(int(np.prod(p.shape)) for p in transfer.trainable_weights)
print(f"Parámetros totales: {transfer.count_params()} · entrenables (solo la cabeza): {n_entrenables}")
h_transfer = entrenar(transfer, epocas=30)
acc_transfer = transfer.evaluate(X_pru, y_pru, verbose=0)[1]
print(f"Transfer learning (base congelada) → exactitud en prueba: {acc_transfer:.3f}")

---

## 🏁 Fin de la parte de clase

- Lo que queda (los ejercicios 🏠 y el mini-reto) está en el **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/para_casa/S11_02_transfer_learning.ipynb)**.
- Ahí también tienes, en cada ejercicio de casa, su celda plegada «🆘 Solución de referencia» por si te quedas atascado.
